# Flows

This chapter joins TTWA centroids with cleaned OD edges to build an `sfnetwork` and draws the full commuting network in monochrome with edge transparency proportional to flow volume.

**Previous:** [Nodes](04_nodes.ipynb)  
**Next:** [Partition](06_partition.ipynb)


In [1]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(sf)
  library(sfnetworks)
  library(tidygraph)
  library(ggraph)
  library(ggplot2)
  library(here)
})


In [2]:
proj_dir <- here::here("projects", "uk-urban-systems-network")
data_dir <- file.path(proj_dir, "data")
fig_dir <- file.path(proj_dir, "figures")
dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(fig_dir, recursive = TRUE, showWarnings = FALSE)


In [3]:
ttwa_centroids <- readRDS(file.path(data_dir, "ttwa_centroids.rds"))
od_ttwa_pairs <- readRDS(file.path(data_dir, "od_ttwa_pairs.rds"))


## Build commuting graph


In [4]:
nodes_sf <- ttwa_centroids |>
  mutate(name = ttwa11cd)

coords <- st_coordinates(nodes_sf)
nodes_tbl <- nodes_sf |>
  st_drop_geometry() |>
  mutate(x = coords[, 1], y = coords[, 2])

edge_tbl <- od_ttwa_pairs |>
  transmute(from = origin_ttwa, to = dest_ttwa, flow)

g_net <- tbl_graph(nodes = nodes_tbl, edges = edge_tbl, directed = TRUE)

network_bundle <- list(
  graph = g_net,
  nodes_sf = nodes_sf,
  edges = edge_tbl
)

saveRDS(network_bundle, file.path(data_dir, "sfnetwork_graph.rds"))


## Network map

In [5]:
layout_manual <- create_layout(
  g_net,
  layout = "manual",
  x = nodes_tbl$x,
  y = nodes_tbl$y
)

p_flows <- ggraph(layout_manual) +
  geom_edge_link(aes(alpha = flow), colour = "grey20", show.legend = FALSE) +
  geom_node_point(size = 1.5, colour = "grey10") +
  scale_edge_alpha_continuous(range = c(0.05, 0.9)) +
  coord_sf(crs = sf::st_crs(nodes_sf), default_crs = sf::st_crs(nodes_sf)) +
  labs(
    title = "TTWA commuting network",
    subtitle = "Edge alpha proportional to flow volume (monochrome)"
  ) +
  theme_void()

fig_flows <- file.path(fig_dir, "05_flows_network.png")
ggsave(fig_flows, p_flows, width = 9, height = 10, dpi = 300)
invisible(fig_flows)


[1] "/Users/areeslindley/Documents/Git_repositories/projects-website/projects/uk-urban-systems-network/figures/05_flows_network.png"

![Commuting flow network](figures/05_flows_network.png)

---

**Next:** [Partition →](06_partition.ipynb)
